# CSE476: Agentic AI and Intelligent Automation
## Task 2 (Project 2 / Q17): Library Acquisition and Collection Development Committee

**Academic Context:**
- **Course Code:** CSE476 — Agentic AI and Intelligent Automation
- **Project Topic:** Library Acquisition and Collection Development Committee (Q17)
- **Evaluation Units:** Unit III (Agent Workflows & Architectures) and Unit IV (Orchestration, RAG, & Memory)
- **Maximum Marks:** 30
- **Student Group:** 
  - Priya Rawat (Reg. No: 12314866)
  - Chiluveru Shiva Prasad (Reg. No: 12304258)

### Project Objective
Automate university library book acquisitions under a fixed operational budget of **₹20,000.00**.
Given requests from faculty, students, or departments, the system deploys a **five-agent committee** coordinated via **Microsoft Agent Framework** using **fan-out / fan-in** orchestration, **local RAG** over acquisition policy rules, and a **persistent shared memory** blackboard to render one of three deterministic outcomes:
- **BUY**: Book fulfills academic policy, demonstrates HIGH patron demand, and fits available budget.
- **DEFER**: Academic value confirmed but demand is low/medium, budget is temporarily insufficient, or evidence is incomplete.
- **REJECT**: Request violates collection policy (e.g. non-academic leisure material) or is an existing duplicate in catalog.

## 2. Technology Stack

| Component | Technology | Role in System |
|---|---|---|
| **Language** | Python 3.12+ | Core programming runtime |
| **Multi-Agent Framework** | `agent-framework-core` & `agent-framework-openai` | Microsoft Agent Framework orchestration, executors, and graph |
| **LLM Provider** | Groq API (`qwen/qwen3.8-27b`) | Natural language synthesis, policy explanation, and committee reports |
| **Local RAG** | `scikit-learn` (TF-IDF & Cosine Similarity) | Grounded policy retrieval without paid external embedding APIs |
| **Financial Arithmetic** | Python `Decimal` | Exact currency math preventing floating-point rounding errors |
| **Data Storage** | JSON & CSV | Synthetic circulation history and policy document store |
| **Shared Memory** | `SharedMemoryService` with JSON persistence | Application blackboard accessible across all 5 agents |
| **Evaluation & Tests** | `pytest` & `pandas` | 10-case evaluation benchmark with 100% accuracy target |
| **Frontend** | Streamlit | University committee demonstration dashboard |

## 3. Multi-Agent Architecture Diagram

```text
                           USER BOOK REQUEST
                                  |
                                  v
                      +-----------------------+
                      | REQUEST INTAKE AGENT  |
                      +-----------------------+
                                  |
                           VALIDATED REQUEST
                                  |
                     +------------+------------+
                     | (Fan-Out)  |            |
                     v            v            v
             +------------+ +------------+ +------------+
             |   POLICY   | |   DEMAND   | |   BUDGET   |
             |   AGENT    | |   AGENT    | |   AGENT    |
             | (Local RAG)| |(Circulation| |(Decimal    |
             |            | | Statistics)| | Calculator)|
             +------------+ +------------+ +------------+
                     |            |            |
                     +------------+------------+
                                  |
                           (Fan-In Barrier)
                                  v
                     +-------------------------+
                     |   ACQUISITIONS MANAGER  |
                     +-------------------------+
                                  |
                         DETERMINISTIC RULES
                                  |
                                  v
                     +-------------------------+
                     |  FINAL DECISION RECORD  |
                     |  (BUY / DEFER / REJECT) |
                     +-------------------------+
                                  |
                                  v
                     +-------------------------+
                     |  SHARED MEMORY & BUDGET |
                     +-------------------------+
```

## 4. Explanation of the Five Specialized Agents

1. **Request Intake Agent (`RequestIntakeAgent`):**
   - Ingests raw book requests (title, requester, price, optional category).
   - Validates mandatory fields (title non-empty, price > 0, requester present).
   - Assigns unique Request IDs and produces structured request records in shared memory.

2. **Collection Policy Retrieval Agent (`PolicyRetrievalAgent`):**
   - Executes local RAG vector search over 12 acquisition policy markdown documents.
   - Computes cosine similarity to detect academic alignment, duplicate rules, or scope violations.
   - Grounded strictly in retrieved evidence; never invents regulatory clauses.

3. **Demand Agent (`DemandAgent`):**
   - Accesses historical circulation records from the library dataset.
   - Computes weighted demand using the statistics tool:
     $$\text{Demand Score} = (\text{Checkout Score} \times 0.70) + (\text{Reservation Score} \times 0.30)$$
   - Categorizes demand as **HIGH** ($\ge 70.0$), **MEDIUM** ($40.0 - 69.99$), or **LOW** ($< 40.0$).

4. **Budget Agent (`BudgetAgent`):**
   - Inspects the current library budget balance using Python `Decimal`.
   - Evaluates whether the requested price is affordable without exceeding available funds.
   - Does *not* independently deduct funds before final manager validation.

5. **Acquisitions Manager (`AcquisitionsManager`):**
   - Synchronizes findings from Policy, Demand, and Budget agents at the fan-in barrier.
   - Applies deterministic precedence rules to yield the final **BUY**, **DEFER**, or **REJECT** decision.
   - Updates remaining budget only for verified BUY acquisitions.

## 5. Explanation of Fan-Out and Fan-In

- **Fan-Out (`WorkflowBuilder.add_fan_out_edges`):**
  Once the Request Intake Agent produces a validated request, the message is concurrently broadcast to three independent specialist executors: Policy, Demand, and Budget. Because none of these three agents depends on the output of the others, they execute concurrently without sequential bottlenecks.

- **Fan-In (`WorkflowBuilder.add_fan_in_edges`):**
  The workflow establishes a synchronization barrier before the Acquisitions Manager. The Manager executor is invoked only when all three specialist results are fully received. This prevents race conditions and ensures financial or policy decisions are never made on partial data.

## 6. Environment Setup and Imports
We import the actual project modules from the codebase.

In [1]:
import sys
import os
from decimal import Decimal
from pathlib import Path
import pandas as pd

# Add project root directory to Python path
project_root = Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import actual modules
from memory.shared_memory import SharedMemoryService
from tools.budget_calculator import BudgetCalculator, to_decimal, INITIAL_BUDGET
from tools.demand_statistics import (
    calculate_demand_score,
    classify_demand,
    lookup_circulation_history,
    MAX_CHECKOUTS,
    MAX_RESERVATIONS
)
from rag.document_loader import load_policy_documents
from rag.retrieval import PolicyRetriever
from agents.intake_agent import RequestIntakeAgent
from agents.policy_agent import PolicyRetrievalAgent
from agents.demand_agent import DemandAgent
from agents.budget_agent import BudgetAgent
from agents.acquisition_manager import AcquisitionsManager
from workflows.acquisition_workflow import AcquisitionWorkflow
from evaluation.evaluate import run_evaluation, evaluate_single_request
from agents.llm_client import llm_manager

print("All project modules successfully imported!")

All project modules successfully imported!


## 7. Groq API Connection Test
We test connectivity to the configured Groq model (`qwen/qwen3.8-27b`) or verify deterministic demo fallback.

In [2]:
print(f"LLM Manager Initialized:")
print(f"  Demo Mode Active: {llm_manager.is_demo_mode}")
print(f"  Configured Model: {llm_manager.model}")
print(f"  Base URL: {llm_manager.base_url}")

# Quick synthesis test
test_reply = llm_manager.generate_explanation(
    prompt="Confirm system readiness in one sentence.",
    system_prompt="You are an AI assistant.",
    fallback="System is operational in deterministic mode."
)
print(f"  LLM Test Response: {test_reply}")

LLM Manager Initialized:
  Demo Mode Active: False
  Configured Model: qwen/qwen3.8-27b
  Base URL: https://api.groq.com/openai/v1
  LLM Test Response: I am ready and fully operational to assist you.


## 8. Synthetic Circulation History Dataset Loading
Inspect the historical checkout and reservation data used by the Demand Agent.

In [3]:
circ_csv_path = project_root / "data" / "circulation_history.csv"
df_circ = pd.read_csv(circ_csv_path)

print(f"Total Circulation Records: {len(df_circ)}")
print(df_circ.head(8).to_string(index=False))

Total Circulation Records: 15
                  book_title               category  checkouts  reservations time_period                                                           historical_demand_notes
     Artificial Intelligence       Computer Science         90            27   2025-2026          Consistently high demand across 4 CSE batches and AI/ML specializations.
             Data Structures       Computer Science         85            24   2025-2026               Core required 2nd year B.Tech subject text with constant waitlists.
     Rare Academic Reference   Reference & Archives         25             4   2024-2025       Niche historical reference monograph with sporadic graduate student access.
            Database Systems       Computer Science         65            18   2025-2026    Active core textbook; current 3 copies already in central circulation catalog.
  Cybersecurity Fundamentals          Cybersecurity         88            25   2025-2026        Essential security 

## 9. Policy Document Loading
Load and parse all 12 synthetic acquisition policies from `data/policies/`.

In [4]:
policies = load_policy_documents()
print(f"Total Loaded Policy Documents: {len(policies)}\n")
for p in policies[:5]:
    print(f"[{p.document_id}] {p.title} | Clause: {p.clause} (Category: {p.category})")

Total Loaded Policy Documents: 12

[POL-001] Core Academic Curriculum Alignment and Syllabi Scope | Clause: CLAUSE-SCOPE-1.1 (Category: Curriculum & Scope)
[POL-002] Duplicate Acquisition and Existing Inventory Restrictions | Clause: CLAUSE-DUP-2.1 (Category: Inventory & Duplicates)
[POL-003] Non-Academic, Leisure, and Recreational Exclusions | Clause: CLAUSE-EXCL-3.1 (Category: Collection Scope Restrictions)
[POL-004] Minimum Historical Demand and Circulation Requirements | Clause: CLAUSE-DEM-4.1 (Category: Demand Benchmarks)
[POL-005] Budget Allocation and Non-Deficit Enforcement | Clause: CLAUSE-BUD-5.1 (Category: Budget & Finance)


## 10. Local RAG Retrieval Demonstration
Demonstrate TF-IDF vector retrieval and cosine similarity ranking across sample queries.

In [5]:
retriever = PolicyRetriever()

sample_queries = [
    ("Artificial Intelligence", "Computer Science"),
    ("Existing Database Systems Duplicate", "Computer Science"),
    ("Unrelated Cooking Collection", "Culinary Arts"),
    ("Rare Academic Reference", "Reference & Archives")
]

for title, cat in sample_queries:
    res = retriever.retrieve(title=title, category=cat)
    print(f"Query: '{title}' ({cat})")
    print(f"  -> Retrieved Doc: {res.document_title} [{res.document_id}]")
    print(f"  -> Policy Clause: {res.policy_clause}")
    print(f"  -> Status: {res.policy_status} | Relevance Score: {res.relevance_score}")
    print(f"  -> Explanation: {res.explanation}\n")

Query: 'Artificial Intelligence' (Computer Science)
  -> Retrieved Doc: Core Academic Curriculum Alignment and Syllabi Scope [POL-001]
  -> Policy Clause: CLAUSE-SCOPE-1.1
  -> Status: COMPLIANT | Relevance Score: 0.2477
  -> Explanation: Matches Core Academic Curriculum Alignment and Syllabi Scope under CLAUSE-SCOPE-1.1. Request aligns with academic collection development.

Query: 'Existing Database Systems Duplicate' (Computer Science)
  -> Retrieved Doc: Duplicate Acquisition and Existing Inventory Restrictions [POL-002]
  -> Policy Clause: CLAUSE-DUP-2.1
  -> Status: VIOLATION | Relevance Score: 0.95
  -> Explanation: Duplicate title detected. Policy CLAUSE-DUP-2.1 strictly prohibits duplicate purchases when existing copies are present.

Query: 'Unrelated Cooking Collection' (Culinary Arts)
  -> Retrieved Doc: Non-Academic, Leisure, and Recreational Exclusions [POL-003]
  -> Policy Clause: CLAUSE-EXCL-3.1
  -> Status: VIOLATION | Relevance Score: 0.92
  -> Explanation: Non-academic

## 11. Shared Memory Initialisation
Initialize the thread-safe persistent application memory blackboard.

In [6]:
memory = SharedMemoryService()
memory.reset()
print("Shared Memory Initialized:")
print(f"  Initial Budget: ₹{memory.initial_budget}")
print(f"  Remaining Budget: ₹{memory.get_remaining_budget()}")
print(f"  Total Approved Expenditure: ₹{memory.get_total_approved_expenditure()}")
print(f"  Validated Requests: {len(memory.validated_requests)}")

Shared Memory Initialized:
  Initial Budget: ₹20000.00
  Remaining Budget: ₹20000.00
  Total Approved Expenditure: ₹0.00
  Validated Requests: 0


## 12. Request Intake Agent Execution
Test parsing and validation of valid vs invalid book requests.

In [7]:
intake_agent = RequestIntakeAgent(memory_service=memory)

# 1. Valid Request
valid_input = {
    "request_id": "DEMO-01",
    "book_title": "Deep Learning Systems",
    "requester": "CSE Dept",
    "price": 2800.0,
    "category": "Computer Science"
}
val_res = intake_agent.process_request(valid_input)
print(f"Valid Request Processed: {val_res}")

# 2. Invalid Request (Missing Title)
try:
    intake_agent.process_request({"requester": "Faculty", "price": 1500.0})
except ValueError as e:
    print(f"Caught Expected Validation Error: {e}")

Valid Request Processed: {'request_id': 'DEMO-01', 'book_title': 'Deep Learning Systems', 'requester': 'CSE Dept', 'price': '2800.00', 'category': 'Computer Science', 'status': 'VALIDATED'}
Caught Expected Validation Error: Validation failed: Book title is mandatory and cannot be empty.


## 13. Policy Retrieval Agent Execution
Evaluate curricular alignment and compliance using the RAG Policy Agent.

In [8]:
policy_agent = PolicyRetrievalAgent(retriever=retriever, memory_service=memory)
policy_findings = policy_agent.evaluate_policy(val_res)
print("Policy Agent Findings:")
for k, v in policy_findings.items():
    print(f"  {k}: {v}")

Policy Agent Findings:
  agent: Collection Policy Retrieval Agent
  request_id: DEMO-01
  policy_document_id: POL-001
  policy_document_title: Core Academic Curriculum Alignment and Syllabi Scope
  policy_clause: CLAUSE-SCOPE-1.1
  relevance_score: 0.1844
  policy_status: COMPLIANT
  retrieval_confidence: MEDIUM
  evidence: Matches Core Academic Curriculum Alignment and Syllabi Scope under CLAUSE-SCOPE-1.1. Request aligns with academic collection development.
  explanation: The acquisition of 'Deep Learning Systems' is compliant with CLAUSE-SCOPE-1.1, as it aligns with the Core Academic Curriculum Alignment and Syllabi Scope for the Computer Science category.


## 14. Demand Agent Execution
Calculate reader demand using synthetic circulation data and the statistical tool.

In [9]:
demand_agent = DemandAgent(memory_service=memory)
demand_findings = demand_agent.evaluate_demand(val_res)
print("Demand Agent Findings:")
for k, v in demand_findings.items():
    print(f"  {k}: {v}")

Demand Agent Findings:
  agent: Demand Agent
  request_id: DEMO-01
  demand_score: 0.0
  demand_level: LOW
  checkouts: 0
  reservations: 0
  evidence: No historical circulation or reservation records found in library catalog.
  data_status: INSUFFICIENT_DATA
  explanation: Insufficient circulation data to verify patron demand.


## 15. Budget Agent Execution
Assess affordability using the Budget Agent and verify that money has not yet been deducted.

In [10]:
calc = BudgetCalculator(initial_budget=INITIAL_BUDGET)
budget_agent = BudgetAgent(calculator=calc, memory_service=memory)
budget_findings = budget_agent.evaluate_budget(val_res)

print("Budget Agent Findings:")
for k, v in budget_findings.items():
    print(f"  {k}: {v}")
print(f"Confirm No Deduction Occurred Yet: Current Balance = ₹{calc.get_remaining_budget()}")

Budget Agent Findings:
  agent: Budget Agent
  request_id: DEMO-01
  requested_price: 2800.00
  current_balance: 20000.00
  affordable: True
  projected_remaining: 17200.00
  deficit: 0.00
  explanation: The purchase of 'Deep Learning Systems' is fiscally affordable, as the ₹2,800.00 cost leaves a positive projected balance of ₹17,200.00 from the current ₹20,000.00 funds.
Confirm No Deduction Occurred Yet: Current Balance = ₹20000.00


## 16. Fan-Out / Fan-In Microsoft Agent Framework Workflow
Examine the workflow graph topology and executor bindings.

In [11]:
workflow = AcquisitionWorkflow(memory_service=memory, calculator=calc)
print("Microsoft Agent Framework Workflow Initialized:")
print(f"  Executors in Graph: {list(workflow.executors.keys())}")
print(f"  Start Executor: {workflow.executors['intake'].id}")
print("  Graph Topology: intake -> [policy, demand, budget] (fan-out) -> manager (fan-in)")

Microsoft Agent Framework Workflow Initialized:
  Executors in Graph: ['intake', 'policy', 'demand', 'budget', 'manager']
  Start Executor: intake
  Graph Topology: intake -> [policy, demand, budget] (fan-out) -> manager (fan-in)


## 17. Acquisitions Manager Execution
The Acquisitions Manager evaluates deterministic precedence and applies approved purchase deductions.

In [12]:
manager_agent = AcquisitionsManager(memory_service=memory, calculator=calc)
decision_res = manager_agent.make_decision(
    request_data=val_res,
    policy_findings=policy_findings,
    demand_findings=demand_findings,
    budget_findings=budget_findings
)
print("Acquisitions Manager Final Output:")
print(f"  Decision: {decision_res['decision']}")
print(f"  Clause: {decision_res['policy_clause']}")
print(f"  Reason: {decision_res['decision_reason']}")
print(f"  Remaining Budget After Purchase: ₹{decision_res['remaining_budget']}")

Acquisitions Manager Final Output:
  Decision: DEFER
  Clause: CLAUSE-SCOPE-1.1
  Reason: Demand score (0.0/100, LOW) does not meet the mandatory HIGH demand threshold (>= 70.0). Deferred for demand monitoring.
  Remaining Budget After Purchase: ₹20000.00


## 18. Complete End-to-End Workflow Example
Execute a full end-to-end request through the Microsoft Agent Framework graph.

In [13]:
# Reset memory and calculator for clean demonstration
memory.reset()
calc.reset()
workflow = AcquisitionWorkflow(memory_service=memory, calculator=calc)

req_e2e = {
    "request_id": "R01",
    "book_title": "Artificial Intelligence",
    "requester": "CSE Department",
    "price": 2500.0,
    "category": "Computer Science"
}

e2e_res = workflow.run(req_e2e)
print("=== END-TO-END WORKFLOW RESULT ===")
for k, v in e2e_res.items():
    print(f"  {k}: {v}")

=== END-TO-END WORKFLOW RESULT ===
  request_id: R01
  book_title: Artificial Intelligence
  requester: CSE Department
  price: 2500.00
  policy_status: COMPLIANT
  policy_clause: CLAUSE-SCOPE-1.1
  policy_document_id: POL-001
  policy_evidence: Matches Core Academic Curriculum Alignment and Syllabi Scope under CLAUSE-SCOPE-1.1. Request aligns with academic collection development.
  demand_score: 90.0
  demand_level: HIGH
  demand_evidence: Consistently high demand across 4 CSE batches and AI/ML specializations.
  budget_available: 20000.00
  decision: BUY
  decision_reason: Compliant with CLAUSE-SCOPE-1.1, verified HIGH demand (90.0/100), and fully affordable within remaining budget.
  executive_summary: The Committee approves the acquisition of 'Artificial Intelligence' for the CSE Department, as it complies with CLAUSE-SCOPE-1.1, demonstrates high demand (90.0/100), and fits within the remaining budget.
  remaining_budget: 17500.00
  execution_time_seconds: 1.501


## 19. Execution of Exactly Ten Evaluation Cases
Process all 10 baseline evaluation requests sequentially to test budget tracking and decision precedence.

In [14]:
df_eval, eval_summary = run_evaluation(reset_state=True)
print(f"Successfully evaluated all {eval_summary['total_cases']} benchmark cases.")

Successfully evaluated all 10 benchmark cases.


## 20. Expected versus Actual Decision Comparison Table
Compare the actual decisions against the academic ground truth baseline.

In [15]:
display_cols = [
    "request_id", "book_title", "category", "price",
    "expected_decision", "actual_decision", "status",
    "budget_before", "budget_after", "policy_clause", "demand_level"
]
print(df_eval[display_cols].to_string(index=False))

request_id                          book_title               category    price expected_decision actual_decision status budget_before budget_after    policy_clause demand_level
       R01             Artificial Intelligence       Computer Science ₹2500.00               BUY             BUY   PASS     ₹20000.00    ₹17500.00 CLAUSE-SCOPE-1.1         HIGH
       R02                     Data Structures       Computer Science ₹1800.00               BUY             BUY   PASS     ₹17500.00    ₹15700.00 CLAUSE-PROG-11.1         HIGH
       R03             Rare Academic Reference   Reference & Archives ₹3000.00             DEFER           DEFER   PASS     ₹15700.00    ₹15700.00  CLAUSE-REF-10.1          LOW
       R04 Existing Database Systems Duplicate       Computer Science ₹2200.00            REJECT          REJECT   PASS     ₹15700.00    ₹15700.00   CLAUSE-DUP-2.1       MEDIUM
       R05          Cybersecurity Fundamentals          Cybersecurity ₹4500.00               BUY             BUY   

## 21. Decision Accuracy Calculation
Verify exact accuracy against the required 10-case evaluation metric.

In [16]:
acc = eval_summary["accuracy_percent"]
passed = eval_summary["passed_cases"]
total = eval_summary["total_cases"]

print("=" * 60)
print(f"DECISION ACCURACY: {acc}% ({passed}/{total} Test Cases Passed)")
print("=" * 60)
assert acc == 100.0, "Evaluation failed: accuracy must be 100%!"

DECISION ACCURACY: 100.0% (10/10 Test Cases Passed)


## 22. Budget Audit and Validation
Confirm that approved purchases total ₹16,800.00 and remaining budget equals ₹3,200.00.

In [17]:
spent = eval_summary["total_approved_expenditure"]
remaining = eval_summary["final_remaining_budget"]
expected_spent = eval_summary["expected_expenditure"]
expected_remaining = eval_summary["expected_remaining"]

print("=" * 60)
print("BUDGET INTEGRITY AUDIT")
print("=" * 60)
print(f"Initial Budget:              ₹{INITIAL_BUDGET}")
print(f"Total Approved Expenditure:  {spent} (Expected: {expected_spent})")
print(f"Final Remaining Budget:      {remaining} (Expected: {expected_remaining})")
print(f"Mathematical Integrity:      {'VALID' if eval_summary['budget_integrity_valid'] else 'INVALID'}")
print(f"Sequential Budget Match:     {eval_summary['matches_ground_truth']}")
print("=" * 60)

assert spent == "₹16800.00", f"Expenditure mismatch: {spent}"
assert remaining == "₹3200.00", f"Remaining budget mismatch: {remaining}"

BUDGET INTEGRITY AUDIT
Initial Budget:              ₹20000.00
Total Approved Expenditure:  ₹16800.00 (Expected: ₹16800.00)
Final Remaining Budget:      ₹3200.00 (Expected: ₹3200.00)
Mathematical Integrity:      VALID
Sequential Budget Match:     True


## 23. Final Evaluation Summary

### Verification Summary Table

| Evaluation Requirement | Expected Target | Actual Result | Verification Status |
|---|---|---|---|
| **Five Distinct Agents** | Separate modules for all 5 roles | 5 dedicated modules in `agents/` | **PASSED** |
| **Agent Framework** | Microsoft Agent Framework `WorkflowBuilder` | Real executors, fan-out/fan-in graph | **PASSED** |
| **Local RAG** | TF-IDF & Cosine Similarity over local policies | 12 policy docs indexed with scikit-learn | **PASSED** |
| **Shared Memory** | Persistent JSON blackboard | `SharedMemoryService` with live trace | **PASSED** |
| **10 Evaluation Cases** | Exact baseline cases R01 to R10 | All 10 executed sequentially | **PASSED** |
| **Decision Accuracy** | 100% agreement with expected outcomes | **100.0% (10/10 PASS)** | **PASSED** |
| **Approved Expenditure** | Exactly ₹16,800.00 | **₹16,800.00** | **PASSED** |
| **Final Remaining Budget**| Exactly ₹3,200.00 | **₹3,200.00** | **PASSED** |
| **Non-Negative Budget** | Budget never drops below zero | Validated via Python `Decimal` | **PASSED** |

### Conclusion
The Library Acquisition and Collection Development Committee multi-agent system fully satisfies all requirements of Project 2 / Q17 for CSE476. The system demonstrates robust multi-agent orchestration, grounded retrieval, explainable decision precedence, and strict fiscal integrity.